# CivicLens AI — YOLO11 training on Kaggle (free GPU)

This notebook runs the repository's reproducible YOLO11n training pipeline on Kaggle's free GPU when your account has accelerator quota.

**Before running:** In Kaggle, open Notebook settings and select **GPU** (T4/P100 if offered) and turn **Internet** on. Run cells from top to bottom. Training may take several hours.

This notebook does not claim training is complete until the final evaluation and artifact checks pass.

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU is not enabled or unavailable. In Notebook settings, select GPU and retry.")

In [ ]:
!git clone https://github.com/ayush0531/civiclens-ai.git
%cd /kaggle/working/civiclens-ai
!git checkout main
!pip install -q -r ml/requirements.txt

## Download and extract RDD2022 India

The published archive is large (about 12.7 GB). This cell extracts the nested India archive and deletes the outer archive to conserve Kaggle disk space. The labelled India images are then extracted for conversion.

In [ ]:
from pathlib import Path
import urllib.request, zipfile, shutil, os

outer = Path("/kaggle/working/RDD2022_full.zip")
india_zip = Path("/kaggle/working/India.zip")
india_root = Path("/kaggle/working/RDD2022_India")
url = "https://ndownloader.figshare.com/files/38030910"

if not india_zip.exists() and not india_root.exists():
    print("Downloading RDD2022 archive. This may take a while...")
    urllib.request.urlretrieve(url, outer)
    print(f"Downloaded {outer.stat().st_size / (1024**3):.2f} GiB")
    with zipfile.ZipFile(outer) as z:
        member = next((n for n in z.namelist() if n.lower().endswith("/india.zip") or n.lower() == "india.zip"), None)
        if member is None:
            raise RuntimeError("India.zip not found in the RDD2022 archive.")
        with z.open(member) as src, india_zip.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    outer.unlink(missing_ok=True)

if not india_root.exists():
    india_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(india_zip) as z:
        z.extractall(india_root)
    india_zip.unlink(missing_ok=True)

xml_count = sum(1 for _ in india_root.rglob("*.xml"))
image_count = sum(1 for p in india_root.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print(f"India images: {image_count}; XML annotations: {xml_count}")
if xml_count < 100:
    raise RuntimeError("Too few labelled India XML annotations were extracted; inspect the dataset archive.")

## Prepare YOLO-format dataset

The preparation script pairs XML annotations with images by basename, creates a deterministic 80/10/10 split, and maps the four RDD2022 classes.

In [ ]:
!python ml/prepare_rdd2022.py --source /kaggle/working/RDD2022_India --output /kaggle/working/datasets/rdd2022-india

## Train YOLO11n

The default is 100 epochs at 640px. If the Kaggle session has limited remaining GPU time, you may reduce epochs, but clearly report the actual number completed and do not present a shortened run as a 100-epoch run.

In [ ]:
EPOCHS = 100
BATCH = 16
!python ml/train_yolo11.py --data /kaggle/working/datasets/rdd2022-india/data.yaml --epochs {EPOCHS} --imgsz 640 --batch {BATCH} --device 0 --workers 4

## Review evaluation and plots

Training is considered successful only if the script produced a credible best.pt, held-out metrics, an ONNX export, and a passing ONNX Runtime smoke test.

In [ ]:
import json
from IPython.display import display, Image
runs = Path("/kaggle/working/civiclens-ai/runs")
summaries = sorted(runs.rglob("evaluation_summary.json"), key=lambda p: p.stat().st_mtime)
if not summaries:
    raise RuntimeError("No evaluation_summary.json found. Training/evaluation/export did not finish.")
summary_path = summaries[-1]
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))
run_dir = summary_path.parent
for name in ["confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png", "F1_curve.png", "results.png"]:
    p = run_dir / name
    if p.exists():
        print("Plot:", name)
        display(Image(filename=str(p)))

## Package downloadable training artifacts

The ZIP contains the real checkpoint(s), exported ONNX, evaluation JSON, CSV logs, and plots. Download it from the Kaggle notebook output after the cell completes.

In [ ]:
import zipfile
artifact_zip = Path("/kaggle/working/CivicLens_YOLO11_Training_Artifacts.zip")
with zipfile.ZipFile(artifact_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in run_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".pt", ".onnx", ".json", ".csv", ".png"}:
            z.write(p, p.relative_to(run_dir.parent))
print("Artifact archive:", artifact_zip)
print("Size (MiB):", round(artifact_zip.stat().st_size / (1024**2), 2))
print("TRAINING_AND_ONNX_SMOKE_TEST=PASS")
print("Important: held-out RDD2022 metrics are not independent Nagpur field validation.")

## Important limitations

- Do not upload the raw RDD2022 dataset to the public source repository.
- The metrics are from a held-out split of labelled RDD2022 India images, not independent Nagpur road photographs.
- Review class-wise metrics, confusion matrix, prediction examples, and false positives before deployment.
- Do not enable production model flags until the ONNX input/output contract is verified against the backend and local field images are evaluated.
- Keep the Kaggle notebook output ZIP as the source of the trained model artifacts.